# 규칙 기반 Trigger — 파일 업로드 후 자동 실행

**Colab에서 ‘런타임 → 모두 실행’ 후 `best.pt`와 영상을 차례로 업로드하면 됩니다.**
영상은 여러 개를 선택할 수 있으며 **객체 검출 → 추적 → 위험 판단 → 결과 저장**이 자동으로 수행됩니다.
프레임이나 검출 결과를 사용자가 직접 만들거나 입력할 필요가 없습니다.

- 기존에 확정한 규칙과 하이퍼파라미터를 그대로 사용합니다.
- 영상별로 검출·추적 표시가 있는 `annotated.mp4`, 이벤트 로그, 요약을 저장하고 완료 후 ZIP 다운로드를 시작합니다.
- 영상의 모든 프레임을 순서대로 분석합니다. 업로드 영상 분석용이며 카메라 실시간 송출 UI는 포함하지 않습니다.
- **VLM 담당자는 3번 연결 셀에 기존 함수를 등록하면 됩니다.** 등록 전에는 Trigger까지만 실행합니다.

결과 영상에는 원본 음성이 포함되지 않습니다. 원본 영상과 가중치는 결과 ZIP에서 제외합니다.
중간 로그는 Colab의 `/content/trigger_outputs`에 저장되므로 런타임이 초기화되기 전에 다운로드하세요.


## 1. 실행 환경 준비


In [ ]:
%pip install -q "ultralytics==8.4.175" "lap>=0.5.12" opencv-python numpy


## 2. 확정 규칙 불러오기

아래 네 코드 셀은 전달한 최종 규칙 코드와 동일합니다.
일반 접근·차량 급진입·신호등 red↔green 변화를 감지합니다.
`person`은 알림 대상이 아니며, `stairs`는 장애물 접근 규칙을 따릅니다.
별도의 보행경로 ROI·단차 높이·계단 방향 판정은 포함하지 않습니다.
학습 모델의 클래스에 `traffic_light`(또는 `traffic light`)가 있어야 하며, 클래스 번호는 모델에서 자동으로 읽습니다.


In [ ]:
from collections import defaultdict, deque
from types import MappingProxyType

import cv2
import numpy as np

In [ ]:
# Test Recall 46.67%를 기록한 기존 규칙의 확정값입니다.
# MappingProxyType은 의도하지 않은 항목 변경을 막습니다.
FINAL_PARAMS = MappingProxyType({
    "GROWTH_THRESHOLD": 1.10,
    "CENTER_SPEED_MAX": 0.08,
    "TRIGGER_STREAK": 10,
    "RAPID_GROWTH_THRESHOLD": 2.00,
    "RAPID_CENTER_SPEED_MIN": 0.10,
    "RAPID_TRIGGER_STREAK": 3,
    "APPROACH_WINDOW": 9,
    "APPROACH_AVG": 3,
    "MAX_OBSERVATION_GAP": 0.30,
})

VEHICLE_NAMES = (
    "bicycle", "car", "motorcycle", "bus", "truck", "other_vehicle", "scooter",
)
OBSTACLE_NAMES = (
    "bench", "potted_plant", "chair", "obstacle", "stairs", "tree",
)

# YOLO / BYTETrack 연결부에 전달할 기존 검출 설정
DETECTOR_SETTINGS = MappingProxyType({
    "conf": 0.5,
    "imgsz": 640,
    "tracker": "bytetrack.yaml",
    "persist": True,
    "verbose": False,
})

# 기존 카메라 움직임 보정(GMC) 설정
GMC_IMAGE_SIZE = 640
GMC_MIN_POINTS = 20
GMC_MIN_INLIER_RATIO = 0.6
GMC_MAX_FB_ERROR = 1.5

In [ ]:
def classify_light_color(frame, box):
    x1, y1, x2, y2 = map(int, box)
    height, width = frame.shape[:2]

    x1 = max(0, min(x1, width))
    x2 = max(0, min(x2, width))
    y1 = max(0, min(y1, height))
    y2 = max(0, min(y2, height))

    roi = frame[y1:y2, x1:x2]
    if roi.size == 0:
        return "unknown"

    hsv = cv2.cvtColor(roi, cv2.COLOR_BGR2HSV)
    h, s, v = cv2.split(hsv)

    red_mask = ((h < 10) | (h > 170)) & (s > 70) & (v > 100)
    green_mask = (h > 35) & (h < 90) & (s > 50) & (v > 80)

    red_score = red_mask.sum()
    green_score = green_mask.sum()
    min_pixels = max(3, int(roi.shape[0] * roi.shape[1] * 0.005))

    if max(red_score, green_score) < min_pixels:
        return "unknown"
    if red_score > green_score * 1.2:
        return "red"
    if green_score > red_score * 1.2:
        return "green"
    return "unknown"


class BackgroundMotion:
    def __init__(self):
        self.prev_gray = None
        self.prev_mask = None

    def update(self, frame, boxes):
        h, w = frame.shape[:2]
        scale = min(1.0, GMC_IMAGE_SIZE / max(h, w))
        sw = max(2, round(w * scale))
        sh = max(2, round(h * scale))

        gray = cv2.cvtColor(
            cv2.resize(frame, (sw, sh)), cv2.COLOR_BGR2GRAY
        )
        mask = np.full((sh, sw), 255, dtype=np.uint8)
        sx, sy = sw / w, sh / h

        for box in boxes:
            x1, y1, x2, y2 = map(float, box)
            margin = 5
            left = max(0, int(np.floor(x1 * sx)) - margin)
            top = max(0, int(np.floor(y1 * sy)) - margin)
            right = min(sw, int(np.ceil(x2 * sx)) + margin)
            bottom = min(sh, int(np.ceil(y2 * sy)) + margin)
            if right > left and bottom > top:
                mask[top:bottom, left:right] = 0

        prev_gray, prev_mask = self.prev_gray, self.prev_mask
        self.prev_gray, self.prev_mask = gray, mask

        info = dict(
            camera_ok=False,
            camera_reason="INIT",
            camera_points=0,
            camera_inliers=0,
            camera_inlier_ratio=0.0,
        )

        if prev_gray is None:
            return None, info
        if prev_gray.shape != gray.shape:
            info["camera_reason"] = "SIZE_CHANGED"
            return None, info

        points0 = cv2.goodFeaturesToTrack(
            prev_gray,
            maxCorners=500,
            qualityLevel=0.01,
            minDistance=10,
            mask=prev_mask,
            blockSize=7,
        )

        if points0 is None or len(points0) < GMC_MIN_POINTS:
            info["camera_reason"] = "FEW_FEATURES"
            return None, info

        points1, status1, _ = cv2.calcOpticalFlowPyrLK(
            prev_gray, gray, points0, None,
            winSize=(21, 21), maxLevel=3,
        )

        if points1 is None or status1 is None:
            info["camera_reason"] = "FLOW_FAILED"
            return None, info

        valid = (
            status1.ravel().astype(bool)
            & np.isfinite(points1.reshape(-1, 2)).all(axis=1)
        )
        p0 = points0.reshape(-1, 2)[valid]
        p1 = points1.reshape(-1, 2)[valid]

        if len(p0) < GMC_MIN_POINTS:
            info["camera_reason"] = "FEW_TRACKS"
            return None, info

        back, status2, _ = cv2.calcOpticalFlowPyrLK(
            gray, prev_gray, p1.reshape(-1, 1, 2), None,
            winSize=(21, 21), maxLevel=3,
        )

        if back is None or status2 is None:
            info["camera_reason"] = "BACK_FLOW_FAILED"
            return None, info

        fb_error = np.linalg.norm(back.reshape(-1, 2) - p0, axis=1)
        valid = (
            status2.ravel().astype(bool)
            & np.isfinite(fb_error)
            & (fb_error <= GMC_MAX_FB_ERROR)
            & (p1[:, 0] >= 0) & (p1[:, 0] < sw)
            & (p1[:, 1] >= 0) & (p1[:, 1] < sh)
        )
        p0, p1 = p0[valid], p1[valid]

        if len(p1):
            indices = np.floor(p1).astype(int)
            background = mask[indices[:, 1], indices[:, 0]] > 0
            p0, p1 = p0[background], p1[background]

        info["camera_points"] = len(p0)

        if len(p0) < GMC_MIN_POINTS:
            info["camera_reason"] = "FEW_BACKGROUND_TRACKS"
            return None, info

        matrix, inliers = cv2.estimateAffinePartial2D(
            p0, p1,
            method=cv2.RANSAC,
            ransacReprojThreshold=2.0,
            maxIters=2000,
            confidence=0.99,
            refineIters=10,
        )

        if matrix is None or inliers is None:
            info["camera_reason"] = "RANSAC_FAILED"
            return None, info

        inlier_mask = inliers.ravel().astype(bool)
        count = int(inlier_mask.sum())
        ratio = count / len(p0)

        info["camera_inliers"] = count
        info["camera_inlier_ratio"] = ratio

        if (
            not np.isfinite(matrix).all()
            or count < GMC_MIN_POINTS
            or ratio < GMC_MIN_INLIER_RATIO
        ):
            info["camera_reason"] = "LOW_CONFIDENCE"
            return None, info

        spread = np.ptp(p0[inlier_mask], axis=0)
        if spread[0] < sw * 0.2 or spread[1] < sh * 0.2:
            info["camera_reason"] = "LOCAL_FEATURES"
            return None, info

        estimated_scale = float(np.hypot(matrix[0, 0], matrix[1, 0]))
        if not 0.9 <= estimated_scale <= 1.1:
            info["camera_reason"] = "EXTREME_SCALE"
            return None, info

        small_transform = np.eye(3)
        small_transform[:2] = matrix
        resize_transform = np.diag([sx, sy, 1.0])
        full_transform = (
            np.linalg.inv(resize_transform)
            @ small_transform
            @ resize_transform
        )

        info["camera_ok"] = True
        info["camera_reason"] = "OK"
        return full_transform[:2], info

In [ ]:
class RuleBasedTrigger:
    """고정 규칙. BGR 프레임과 추적 결과를 받아 이번 프레임의 이벤트만 반환합니다."""

    def __init__(self, class_names, fps):
        names = dict(enumerate(class_names)) if isinstance(class_names, list) else class_names
        self.names = {
            int(k): str(v).strip().lower().replace(" ", "_")
            for k, v in names.items()
        }
        name_to_id = {v: k for k, v in self.names.items()}
        if "traffic_light" not in name_to_id:
            raise ValueError("traffic_light 클래스가 필요합니다.")
        self.vehicle_classes = {name_to_id[n] for n in VEHICLE_NAMES if n in name_to_id}
        self.obstacle_classes = {name_to_id[n] for n in OBSTACLE_NAMES if n in name_to_id}
        self.traffic_light_class = name_to_id["traffic_light"]
        self.target_classes = (
            [name_to_id[n] for n in VEHICLE_NAMES if n in name_to_id]
            + [name_to_id[n] for n in OBSTACLE_NAMES if n in name_to_id]
            + [self.traffic_light_class]
        )
        self.params = FINAL_PARAMS
        self.reset(fps=fps)

    def reset(self, *, fps=None):
        """새 영상/스트림 또는 tracker ID 초기화 시 호출합니다. Tracker도 별도로 초기화하세요."""
        if fps is not None:
            fps = float(fps)
            if not np.isfinite(fps) or fps <= 0:
                raise ValueError("fps는 양수여야 합니다.")
            self.fps = fps
        self.motion = BackgroundMotion()
        self.history = defaultdict(lambda: deque(maxlen=self.params["APPROACH_WINDOW"]))
        self.last_seen = {}
        self.normal_streak = {}
        self.rapid_streak = {}
        self.approach_fired = set()
        self.colors = defaultdict(lambda: deque(maxlen=5))
        self.stable_colors = {}
        self.light_fired = set()
        self._last_frame = 0
        self._last_time = None

    def update(self, frame_bgr, detections, *, frame_index=None, time_s=None):
        """detections: box(xyxy 픽셀), class_id, track_id를 가진 dict의 iterable.

        빈 프레임도 detections=[]로 호출합니다. time_s는 영상/캡처 시각(초)입니다.
        frame_index는 1부터 시작하는 원본 프레임 번호이며 반드시 증가해야 합니다.
        """
        fi = self._last_frame + 1 if frame_index is None else frame_index
        if not isinstance(fi, (int, np.integer)) or fi <= self._last_frame:
            raise ValueError("frame_index는 이전보다 큰 양의 정수여야 합니다.")
        fi = int(fi)
        t = (fi - 1) / self.fps if time_s is None else float(time_s)
        if not np.isfinite(t) or t < 0 or (self._last_time is not None and t <= self._last_time):
            raise ValueError("time_s는 0 이상의 증가하는 영상 시각이어야 합니다.")
        if (not isinstance(frame_bgr, np.ndarray) or frame_bgr.dtype != np.uint8
                or frame_bgr.ndim != 3 or frame_bgr.shape[2] != 3
                or min(frame_bgr.shape[:2]) < 2):
            raise ValueError("frame_bgr는 uint8 H×W×3 BGR 이미지여야 합니다.")
        prepared = []
        for order, detection in enumerate(detections):
            cid = int(detection["class_id"])
            if cid not in self.target_classes:
                continue
            box = np.asarray(detection["box"], dtype=float)
            if (box.shape != (4,) or not np.isfinite(box).all()
                    or box[2] < box[0] or box[3] < box[1]):
                raise ValueError("box는 유한한 [x1, y1, x2, y2] 픽셀 좌표여야 합니다.")
            tid = detection.get("track_id")
            prepared.append(dict(
                box=box.tolist(), class_id=cid,
                **{"class": self.names[cid]},
                track_id=None if tid is None else int(tid),
                order=int(detection.get("order", order)),
                color=(classify_light_color(frame_bgr, box)
                       if cid == self.traffic_light_class else None),
            ))
        matrix, camera = self.motion.update(frame_bgr, [d["box"] for d in prepared])
        height, width = frame_bgr.shape[:2]
        events = self._step(dict(
            frame=fi, time_s=t, width=width, height=height,
            matrix=matrix, camera=camera, detections=prepared,
        ))
        self._last_frame, self._last_time = fi, t
        return events

    def update_yolo(self, frame_bgr, result, *, frame_index=None, time_s=None):
        """Ultralytics model.track(...)[0] 결과를 바로 연결하는 어댑터."""
        detections = []
        if result.boxes is not None:
            for order, box in enumerate(result.boxes):
                detections.append(dict(
                    box=box.xyxy[0].cpu().tolist(),
                    class_id=int(box.cls[0]),
                    track_id=int(box.id[0]) if box.id is not None else None,
                    order=order,
                ))
        return self.update(
            frame_bgr, detections, frame_index=frame_index, time_s=time_s
        )

    def _step(self, observation):
        """동일한 GMC/HSV 관측에 원래 규칙을 순차 적용합니다."""
        p = self.params
        window, avg, gap = p["APPROACH_WINDOW"], p["APPROACH_AVG"], p["MAX_OBSERVATION_GAP"]
        f = observation
        fi, t, camera = f["frame"], f["time_s"], f["camera"]
        if not camera["camera_ok"]:
            self.history.clear()
        else:
            m = np.asarray(f["matrix"], dtype=float)
            for key in list(self.history):
                hist = self.history[key]
                if not hist or (fi - hist[-1]["frame"]) / self.fps > gap:
                    del self.history[key]
                    continue
                for old in hist:
                    old["corrected"] = m[:, :2] @ old["corrected"] + m[:, 2]

        events = []
        for detection in f["detections"]:
            tid, cid = detection["track_id"], detection["class_id"]
            if tid is None:
                continue
            base = dict(
                frame=fi, time_s=t, track_id=tid, class_id=cid,
                **{"class": detection["class"]}, order=detection["order"],
            )
            if cid == self.traffic_light_class:
                key = (cid, tid)
                self.colors[key].append(detection["color"])
                recent = list(self.colors[key])[-3:]
                now = (
                    recent[0] if len(recent) == 3 and len(set(recent)) == 1
                    and recent[0] in ("red", "green") else None
                )
                previous = self.stable_colors.get(key)
                event_key = (cid, tid, previous, now)
                if (now is not None and previous in ("red", "green")
                        and now != previous and event_key not in self.light_fired):
                    self.light_fired.add(event_key)
                    events.append(dict(
                        base, event=f"light_{previous}_to_{now}",
                        trigger_type="light_change",
                        message=f"light ID={tid} {previous}->{now}",
                    ))
                if now is not None:
                    self.stable_colors[key] = now
                continue

            if cid not in self.vehicle_classes and cid not in self.obstacle_classes:
                continue
            x1, y1, x2, y2 = detection["box"]
            center = np.array([(x1 + x2) / 2, (y1 + y2) / 2])
            missing = max(0, fi - self.last_seen[tid] - 1) if tid in self.last_seen else 0
            self.last_seen[tid] = fi
            hist = self.history[tid]
            hist.append(dict(
                frame=fi, time=t, area=max(0, x2 - x1) * max(0, y2 - y1),
                raw=center.copy(), corrected=center.copy(),
            ))
            growth = speed = None
            if len(hist) >= window:
                seq = list(hist)
                past, recent = seq[:avg], seq[-avg:]
                growth = float(
                    np.mean([o["area"] for o in recent])
                    / max(np.mean([o["area"] for o in past]), 1.0)
                )
                dt = max(
                    np.mean([o["time"] for o in recent])
                    - np.mean([o["time"] for o in past]), 1e-6,
                )
                delta = (
                    np.mean([o["corrected"] for o in recent], axis=0)
                    - np.mean([o["corrected"] for o in past], axis=0)
                )
                speed = float(np.linalg.norm(delta / np.array([f["width"], f["height"]])) / dt)
            valid = camera["camera_ok"] and len(hist) >= window
            normal = valid and growth >= p["GROWTH_THRESHOLD"] and speed <= p["CENTER_SPEED_MAX"]
            rapid = (valid and cid in self.vehicle_classes
                     and growth >= p["RAPID_GROWTH_THRESHOLD"]
                     and speed >= p["RAPID_CENTER_SPEED_MIN"])
            ns = (0 if missing else self.normal_streak.get(tid, 0)) + 1 if normal else 0
            rs = (0 if missing else self.rapid_streak.get(tid, 0)) + 1 if rapid else 0
            self.normal_streak[tid], self.rapid_streak[tid] = ns, rs
            if tid not in self.approach_fired and (ns >= p["TRIGGER_STREAK"] or rs >= p["RAPID_TRIGGER_STREAK"]):
                self.approach_fired.add(tid)
                kind = "rapid_entry" if rs >= p["RAPID_TRIGGER_STREAK"] else "approach"
                events.append(dict(
                    base, event="approach", trigger_type=kind,
                    message=f"{detection['class']} ID={tid} {kind} g={growth:.3f} v={speed:.4f}",
                ))
        events.sort(key=lambda event: (event["frame"], event["order"]))
        return events

## 3. 팀의 기존 VLM 함수 연결

**Trigger만 확인할 때는 이 셀을 그대로 실행하세요.**
팀원은 아래 주석의 `forward_to_vlm` 함수 안에 기존 VLM 호출 코드를 넣고, 마지막에 `VLM_CALLBACK = forward_to_vlm`으로 등록하면 됩니다.
그러면 다음 셀에서 위험 이벤트가 발생할 때 자동으로 호출됩니다.

- `event`: 영상명·발생 시각·객체 클래스·추적 ID·위험 유형을 담은 dict.
- `frame_bgr`: 트리거가 발생한 현재 원본 프레임.
- `clip_frames_bgr`: 직전 약 2초에서 초당 최대 4장씩 모은 이미지 목록. 긴 변은 최대 640픽셀입니다.
- VLM이 RGB 이미지를 요구하면 `cv2.cvtColor(image, cv2.COLOR_BGR2RGB)`로 변환하세요.
- 이 예제는 연결 함수가 끝날 때까지 기다립니다. 팀에 비동기 처리 큐가 있다면 연결 함수에서 그 큐에 전달하세요.

기존 VLM 코드가 자신의 영상 버퍼를 사용한다면 `event`의 영상명·시각으로 해당 버퍼를 이용해도 됩니다.


In [ ]:
# VLM 연결 전에는 None을 그대로 둡니다.
VLM_CALLBACK = None

# VLM 담당자가 연결할 때 아래 형태로 등록합니다.
# def forward_to_vlm(event, frame_bgr, clip_frames_bgr):
#     # 여기에 기존 VLM 호출 코드를 넣으세요.
#     # event / 현재 프레임 / 최근 프레임 목록이 자동으로 전달됩니다.
#     pass
#
# VLM_CALLBACK = forward_to_vlm


## 4. 자동 처리 함수 준비

아래 코드는 입력 영상 읽기, YOLO 검출·추적, Trigger 실행, VLM 연결 함수 호출, 결과 저장을 담당합니다.
영상마다 상태를 새로 만들어 서로 다른 영상의 객체 ID와 이력이 섞이지 않도록 합니다.


In [ ]:
from pathlib import Path
from datetime import datetime
from collections import deque
import csv
import gc
import hashlib
import json
import re
import shutil
import traceback
import uuid
from ultralytics import YOLO


def _write_json(path, data):
    Path(path).write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8")


def _preview_frame(frame, result, names, active_events, time_s):
    canvas = frame.copy()
    if result.boxes is not None:
        for box in result.boxes:
            x1, y1, x2, y2 = map(int, box.xyxy[0].cpu().tolist())
            cid = int(box.cls[0])
            tid = int(box.id[0]) if box.id is not None else None
            label = f"{names.get(cid, cid)} ID={tid}"
            cv2.rectangle(canvas, (x1, y1), (x2, y2), (100, 210, 80), 2)
            cv2.putText(canvas, label, (max(0, x1), max(18, y1 - 5)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (100, 210, 80), 1, cv2.LINE_AA)
    lines = [f"Time {time_s:.2f}s"]
    lines += [f"TRIGGER: {e['class']} ID={e['track_id']} {e['trigger_type']}"
              for e in active_events][-4:]
    for i, line in enumerate(lines):
        y = 24 + 24 * i
        cv2.putText(canvas, line, (10, y), cv2.FONT_HERSHEY_SIMPLEX,
                    0.55, (0, 0, 0), 4, cv2.LINE_AA)
        cv2.putText(canvas, line, (10, y), cv2.FONT_HERSHEY_SIMPLEX,
                    0.55, (0, 220, 255), 1, cv2.LINE_AA)
    return canvas


def run_one_video(weights_path, video_path, output_dir, vlm_callback=None):
    """YOLO 검출·추적 → 확정 Trigger → 선택적으로 팀 VLM 호출. 영상마다 상태 초기화."""
    if vlm_callback is not None and not callable(vlm_callback):
        raise TypeError("VLM_CALLBACK은 함수 또는 None이어야 합니다.")
    video_path, output_dir = Path(video_path), Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        cap.release()
        raise ValueError(f"영상을 열 수 없습니다: {video_path.name}")
    writer = None
    model = None
    frame_index = 0
    try:
        fps = float(cap.get(cv2.CAP_PROP_FPS))
        expected_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        if not np.isfinite(fps) or fps <= 0:
            raise ValueError("영상 FPS를 읽을 수 없습니다. 정상적인 MP4 파일로 변환해 주세요.")
        # 영상마다 모델/추적기를 새로 생성해 다른 영상의 ID와 이력이 섞이지 않습니다.
        model = YOLO(str(weights_path))
        if getattr(model, "task", None) not in ("detect", "segment"):
            raise ValueError("객체 검출 또는 세그멘테이션용 best.pt가 필요합니다.")
        trigger = RuleBasedTrigger(model.names, fps)
        print(f"\n분석 시작: {video_path.name} | {fps:.3f} FPS", flush=True)

        # VLM에 전달할 최근 2초의 영상: 최대 초당 4장, 긴 변 640픽셀.
        # 이 버퍼 설정은 검출·추적·Trigger에 사용하는 원본 프레임에 영향을 주지 않습니다.
        clip_seconds, clip_interval, clip_max_side = 2.0, 0.25, 640
        recent_frames = deque(maxlen=12)
        next_clip_time = 0.0
        active_events = []
        event_count = callback_error_count = frames_with_track_ids = 0
        frame_shape = None
        output_video = output_dir / "annotated.mp4"
        fields = ["video", "frame", "time_s", "track_id", "class_id", "class",
                  "event", "trigger_type", "message"]
        with (output_dir / "events.jsonl").open("w", encoding="utf-8") as event_file, \
             (output_dir / "triggers.csv").open("w", encoding="utf-8-sig", newline="") as csv_file, \
             (output_dir / "vlm_errors.jsonl").open("w", encoding="utf-8") as error_file:
            csv_writer = csv.DictWriter(csv_file, fieldnames=fields, extrasaction="ignore")
            csv_writer.writeheader()
            while True:
                ok, frame_bgr = cap.read()
                if not ok:
                    break
                frame_index += 1
                time_s = (frame_index - 1) / fps
                height, width = frame_bgr.shape[:2]
                if frame_shape is None:
                    frame_shape = (height, width)
                    # mp4v는 짝수 해상도가 필요합니다. 저장 영상만 1픽셀까지 패딩합니다.
                    writer = cv2.VideoWriter(str(output_video), cv2.VideoWriter_fourcc(*"mp4v"),
                                             fps, (width + width % 2, height + height % 2))
                    if not writer.isOpened():
                        raise RuntimeError("결과 영상을 저장할 MP4 코덱을 열 수 없습니다.")
                elif frame_bgr.shape[:2] != frame_shape:
                    raise ValueError("영상 도중 해상도가 변경되었습니다.")

                result = model.track(frame_bgr, classes=trigger.target_classes,
                                     **dict(DETECTOR_SETTINGS))[0]
                if result.boxes is not None and any(box.id is not None for box in result.boxes):
                    frames_with_track_ids += 1
                # 검출이 없는 프레임도 호출합니다. 원본 FPS와 프레임 순서를 유지합니다.
                events = trigger.update_yolo(frame_bgr, result,
                                             frame_index=frame_index, time_s=time_s)

                if vlm_callback is not None:
                    if time_s + 1e-9 >= next_clip_time:
                        scale = min(1.0, clip_max_side / max(height, width))
                        small = cv2.resize(frame_bgr, (max(2, round(width * scale)),
                                                       max(2, round(height * scale))))
                        recent_frames.append((time_s, small))
                        next_clip_time = time_s + clip_interval
                    while recent_frames and recent_frames[0][0] < time_s - clip_seconds:
                        recent_frames.popleft()

                for event in events:
                    record = {"video": video_path.name, **event}
                    # VLM 호출보다 먼저 저장해 연결 함수의 실패와 위험 감지 기록을 분리합니다.
                    event_file.write(json.dumps(record, ensure_ascii=False) + "\n")
                    event_file.flush()
                    csv_writer.writerow(record)
                    csv_file.flush()
                    event_count += 1
                    active_events.append(record)
                    print(f"TRIGGER | {video_path.name} | {time_s:.2f}s | {event['message']}", flush=True)
                    if vlm_callback is not None:
                        try:
                            vlm_callback(event=dict(record), frame_bgr=frame_bgr.copy(),
                                         clip_frames_bgr=[img.copy() for _, img in recent_frames])
                        except Exception as exc:
                            callback_error_count += 1
                            error_file.write(json.dumps({"event": record, "error": str(exc),
                                                         "traceback": traceback.format_exc()},
                                                        ensure_ascii=False) + "\n")
                            error_file.flush()
                            print(f"VLM 연결 함수 오류: {exc} — Trigger 분석은 계속합니다.", flush=True)
                active_events = [e for e in active_events if time_s - e["time_s"] <= 2.0]
                preview = _preview_frame(frame_bgr, result, trigger.names, active_events, time_s)
                if width % 2 or height % 2:
                    preview = cv2.copyMakeBorder(preview, 0, height % 2, 0, width % 2,
                                                cv2.BORDER_CONSTANT, value=(0, 0, 0))
                writer.write(preview)
                if frame_index % max(1, round(fps * 5)) == 0:
                    total_text = str(expected_frames) if expected_frames > 0 else "?"
                    print(f"  {video_path.name}: {frame_index}/{total_text} 프레임 | 이벤트 {event_count}개", flush=True)
        if frame_index == 0:
            raise ValueError("읽을 수 있는 영상 프레임이 없습니다.")
        notes = []
        if expected_frames > frame_index + 1:
            notes.append(f"영상 메타데이터는 {expected_frames}프레임이나 {frame_index}프레임만 읽었습니다. 원본 상태를 확인하세요.")
        if frames_with_track_ids == 0:
            notes.append("추적 ID가 생성된 프레임이 없습니다. 검출 결과와 모델 클래스를 확인하세요.")
        summary = dict(video=video_path.name, fps=fps, frames_processed=frame_index,
                       expected_frames=expected_frames, frames_with_track_ids=frames_with_track_ids,
                       trigger_count=event_count, vlm_connected=vlm_callback is not None,
                       vlm_callback_errors=callback_error_count,
                       status="completed_with_notes" if notes else "completed", notes=notes,
                       params=dict(FINAL_PARAMS), detector_settings=dict(DETECTOR_SETTINGS),
                       output_video=str(output_video), events_file=str(output_dir / "events.jsonl"))
        _write_json(output_dir / "summary.json", summary)
        for note in notes:
            print(note, flush=True)
        print(f"완료: {video_path.name} | Trigger {event_count}개 | {output_dir}", flush=True)
        return summary
    except Exception as exc:
        _write_json(output_dir / "error.json", {"video": video_path.name, "error": str(exc),
                                               "frames_processed": frame_index,
                                               "traceback": traceback.format_exc()})
        raise
    finally:
        cap.release()
        if writer is not None:
            writer.release()
        del model
        gc.collect()


def upload_and_run(vlm_callback=None, output_root="/content/trigger_outputs"):
    """이 함수를 실행하면 파일 선택 후 분석까지 자동으로 이어집니다."""
    from google.colab import files
    print("① 학습한 best.pt 파일 1개를 선택하세요.")
    uploaded = files.upload()
    weight_names = [name for name in uploaded if Path(name).suffix.lower() == ".pt"]
    if len(uploaded) != 1 or len(weight_names) != 1:
        raise ValueError("첫 번째 업로드에서는 .pt 가중치 파일 1개만 선택해 주세요.")
    weights_path = Path(weight_names[0]).resolve()
    weights_sha256 = hashlib.sha256(uploaded[weight_names[0]]).hexdigest()
    del uploaded

    print("② 분석할 영상을 선택하세요. 여러 개를 함께 선택할 수 있습니다.")
    uploaded = files.upload()
    extensions = {".mp4", ".mov", ".avi", ".mkv", ".m4v", ".webm"}
    video_paths = [Path(name).resolve() for name in uploaded if Path(name).suffix.lower() in extensions]
    ignored = [name for name in uploaded if Path(name).suffix.lower() not in extensions]
    del uploaded
    if ignored:
        print("영상이 아니어서 제외:", ignored)
    if not video_paths:
        raise ValueError("분석할 영상이 없습니다. MP4 등의 영상 파일을 선택해 주세요.")

    run_dir = Path(output_root) / (datetime.now().strftime("%Y%m%d_%H%M%S") + "_" + uuid.uuid4().hex[:6])
    run_dir.mkdir(parents=True, exist_ok=False)
    _write_json(run_dir / "settings.json", {"weights": weights_path.name, "weights_sha256": weights_sha256,
                                           "params": dict(FINAL_PARAMS), "detector": dict(DETECTOR_SETTINGS),
                                           "vlm_connected": vlm_callback is not None})
    print("업로드 완료. 검출·추적·Trigger 분석을 자동으로 시작합니다.")
    if vlm_callback is None:
        print("현재는 Trigger 결과를 저장합니다. VLM 담당자는 위 연결 셀에 기존 호출 함수를 등록하면 됩니다.")
    summaries = []
    for index, path in enumerate(video_paths, 1):
        safe_stem = re.sub(r"[^\w.-]+", "_", path.stem)[:80] or "video"
        folder = run_dir / f"{index:02d}_{safe_stem}"
        try:
            result = run_one_video(weights_path, path, folder, vlm_callback)
        except Exception as exc:
            print(f"분석 실패: {path.name} | {exc}", flush=True)
            result = {"video": path.name, "status": "failed", "error": str(exc)}
        summaries.append(result)
        _write_json(run_dir / "summary.json", summaries)
    zip_path = Path(shutil.make_archive(str(run_dir), "zip", root_dir=run_dir))
    print(f"\n결과 ZIP: {zip_path}")
    try:
        files.download(str(zip_path))
    except Exception as exc:
        print(f"자동 다운로드를 시작하지 못했습니다: {exc}\n왼쪽 파일 목록에서 위 ZIP을 직접 다운로드하세요.")
    return run_dir, zip_path, summaries


## 5. `best.pt`와 영상 업로드 → 자동 실행

첫 번째 창에서 **가중치 파일 하나**, 두 번째 창에서 **영상 한 개 또는 여러 개**를 선택하세요.
영상 업로드가 끝나면 바로 분석을 시작합니다. 완료되면 결과 ZIP이 다운로드됩니다.
VLM을 연결한 팀 통합 코드에서도 이 셀을 그대로 사용하면 됩니다.


In [ ]:
RUN_DIR, RESULT_ZIP, VIDEO_SUMMARIES = upload_and_run(vlm_callback=VLM_CALLBACK)


## 결과 확인

- `annotated.mp4`: 객체 박스·추적 ID·Trigger 발생 표시가 있는 영상.
- `events.jsonl` / `triggers.csv`: 발생한 이벤트 목록. 이벤트가 없으면 비어 있습니다.
- `summary.json`: 영상별 처리 프레임 수·이벤트 수·처리 상태.
- `vlm_errors.jsonl`: 연결 함수에서 오류가 난 경우의 기록.

가중치나 프레임 처리 조건이 달라지면 이전 평가 수치가 그대로 재현되는 것은 아닙니다.
Train/Test 평가·라벨링·하이퍼파라미터 탐색 코드는 포함하지 않습니다.
